# 355. Design Twitter

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** hash-table, linked-list, design, heap
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/design-twitter/)

Design a simplified version of Twitter where users can post tweets,
follow/unfollow another user, and see the **10 most recent tweet ids** in the
user's news feed.

Implement the `Twitter` class:

- `Twitter()` initializes your twitter object.
- `postTweet(userId, tweetId)` composes a new tweet with ID `tweetId` by the
  user `userId`. Each call is made with a **unique** `tweetId`.
- `getNewsFeed(userId)` retrieves the **10 most recent** tweet ids in the
  user's news feed. Each item must be posted by users the user **followed or
  by the user themself**. Tweets must be **ordered from most recent to least
  recent**.
- `follow(followerId, followeeId)` the user `followerId` starts following the
  user `followeeId`.
- `unfollow(followerId, followeeId)` the user `followerId` stops following the
  user `followeeId`.

---

### Example 1

```
Input:  ["Twitter", "postTweet", "getNewsFeed", "follow", "postTweet", "getNewsFeed", "unfollow", "getNewsFeed"]
        [[],        [1, 5],      [1],           [1, 2],   [2, 6],      [1],           [1, 2],     [1]]
Output: [null,      null,        [5],           null,     null,        [6, 5],        null,       [5]]

Twitter twitter = new Twitter();
twitter.postTweet(1, 5);      // User 1 posts a new tweet (id = 5).
twitter.getNewsFeed(1);       // User 1's news feed: [5]
twitter.follow(1, 2);         // User 1 follows user 2.
twitter.postTweet(2, 6);      // User 2 posts a new tweet (id = 6).
twitter.getNewsFeed(1);       // User 1's news feed: [6, 5] - 6 is newer.
twitter.unfollow(1, 2);       // User 1 unfollows user 2.
twitter.getNewsFeed(1);       // User 1's news feed: [5] again.
```

---

### Constraints

- `1 <= userId, followerId, followeeId <= 500`
- `0 <= tweetId <= 10^4`
- All the tweets have **unique** IDs.
- At most `3 * 10^4` calls will be made to `postTweet`, `getNewsFeed`,
  `follow`, and `unfollow`.
- A user **cannot** follow themself. (LeetCode promises this - but your test
  cell below checks what happens if it *did* happen. Read question 4.)


## Before you write anything

Second design problem - same game as Min Stack: no algorithm over an input,
just a class with promises to keep. But where #155 was one hard operation and
three easy ones, here the work is spread out. Answer these on paper first.

**1.** Four operations. Two of them are a warm-up if you pick the right
container: `follow` and `unfollow` are just "add X to user's collection" and
"remove X from user's collection". Which built-in container makes both O(1)
and refuses duplicates for free? (You used its cousin, the dict, in Two Sum
and Valid Anagram.)

**2.** `getNewsFeed` must order tweets by recency - **across different
users**. User 1 posted, then user 2, then user 1 again. The tweet ids
themselves tell you nothing about *when* (id 5 can be posted after id 900).
So who keeps track of time? What one integer does the `Twitter` object itself
need, and at which moment does it get stamped onto a tweet?

**3.** Where do tweets live? Say each user gets a list and `postTweet`
appends to the end. Now look at one user's list: in what order are the tweets
already sitting there, without you ever sorting? This free ordering is the
key to the whole problem - name it before moving on.

**4.** The feed is "my tweets + my followees' tweets". The tempting shortcut:
make every user follow themself, then the feed is just "followees' tweets".
Trace this sequence with that shortcut in place:

```
postTweet(1, 101)   follow(1, 1)   unfollow(1, 1)   getNewsFeed(1)
```

What did `unfollow(1, 1)` just destroy? How do you include the user's own
tweets so that **no** follow/unfollow call can ever break it?

**5.** `unfollow(1, 2)` when user 1 never followed user 2 - what does your
container's `remove` do there? One of `remove` / `discard` crashes and one
doesn't. Which one do you want?

**6.** The real work: you hold several lists (yours + one per followee),
**each already ordered by time** (question 3), and you must produce the 10
most recent overall. Simplest possible plan: dump them all into one list,
sort by time, take the last 10. That works - what does it cost per
`getNewsFeed` call if the sources hold `T` tweets total? Now the refinement:
each source can contribute at most how many tweets to a top-10? So how much
of each list do you actually need to look at?


## Two routes - A first, B is the classic

**A - collect, sort, slice** *(do this one first)*
One dict `tweets`: userId -> list of `(time, tweetId)` pairs, appended in
post order. One dict `following`: userId -> set of followees. `getNewsFeed`
gathers the lists of the user + followees into one list, sorts by time
descending, returns the first 10 tweet ids. With the constraints capped at
`3 * 10^4` calls this passes easily. Cost per feed: `O(T log T)` for `T`
tweets among the sources. Question 6's refinement - slice each source to its
**last 10** before collecting - caps the sort at `10 * (k+1)` items no matter
how chatty the users are.

**B - merge k sorted lists** *(why this problem sits in the linked-list list)*
Each source list is already sorted by time. Producing "the top 10 across k
sorted lists" without concatenating is exactly LeetCode **#23 Merge k Sorted
Lists**: repeatedly take the newest head among the k lists, step that list
back by one, stop after 10. A **heap** of at most k candidates makes "newest
head" O(log k) per step - `import heapq`, and note `heapq` is a *min*-heap,
so you push `-time` to get newest-first. `O(k + 10 log k)` per feed instead
of sorting everything. Build it only after route A passes - same tests, drop-in
replacement.

Write the class skeleton below. The test runner two cells down replays
LeetCode-style `(ops, args)` sequences against it - same style as Min Stack.


In [62]:
class Node:
    def __init__(self, val, t = 0 , next=None ):
        self.val = val
        self.t = t
        self.next = next

class Stack:
    def __init__(self):
        self.head = None
        self.last = None
    def append(self,node:Node):
        if self.head is None:
            self.head = node
            self.last = node
        else:
            node.next = self.head
            self.head = node

    def add(self,node):
        if self.last is None:
            self.head = node
            self.last = node
        else :
            self.last.next = node
            self.last = node

    def getList(self)->list:
        if self.head :
            curr = self.head
            lst = []
            while curr:
                lst.append(curr.val)
                curr = curr.next
            return lst
        return []
    def getLast10(self):
        s = 0
        if self.head :
            curr = self.head
            lst = []
            while curr and s<10:
                s+=1
                lst.append(curr.val)
                curr = curr.next
            return lst
        return []
    def getListWithT(self):
        s = 0
        if self.head :
            curr = self.head
            lst = []
            while curr and s<10:
                s+=1
                lst.append([curr.val,curr.t])
                curr = curr.next
            return lst
        return []
    def addingStack(self, lst):
        all_nodes = []

        current = self.head
        while current:
            all_nodes.append([current.val, current.t])
            current = current.next

        all_nodes.extend(lst)

        all_nodes.sort(key=lambda x: x[1], reverse=True)

        self.head = None
        self.last = None

        for val, t in all_nodes:
            self.add(Node(val, t))

        return self


    def remove(self , value):
        if self.head is None :
            return
        if self.head.val == value:
            if self.head == self.last :
                self.head = None
                self.last = None
            else : self.head = self.head.next
            return
        if self.last.val == value:
            current = self.head
            while current.next != self.last:
                current = current.next
            current.next = None
            self.last = current
            return
        current = self.head
        prev = self.head
        while current:
            if current.val == value:
                prev.next = current.next
                return
            prev = current
            current = current.next
        return



from typing import Dict
import time

user = {
    1 : {"feeds" : Stack() , "follow" : Stack() , "posts"  : Stack() } ,
    2 : {"feeds" : Stack() , "follow" : Stack() , "posts"  : Stack() }
}



class Twitter:

    def __init__(self):
        self.DB = {}
        self.t = 0
    def postTweet(self, userId: int, tweetId: int) -> None:
        self.t += 1
        tweet = Node(tweetId, self.t)
        feeds = Node(tweetId,self.t)
        # adding the post for the owner
        if userId in self.DB:
            self.DB[userId]["posts"].append(tweet)
        else :
            self.DB[userId] = {"feeds": Stack(), "follow": Stack(), "posts": Stack()}
            self.DB[userId]["posts"].append(tweet)
        self.DB[userId]["feeds"].append(feeds)
        # adding the post to the followers :
        for p in self.DB:
            if userId in  self.DB[p]["follow"].getList():
                self.DB[p]["feeds"].append(Node(tweetId,self.t))

    def getNewsFeed(self, userId: int) -> list:
        if userId in self.DB:
            return self.DB[userId]["feeds"].getLast10()
        return []

    def follow(self, followerId: int, followeeId: int) -> None:
        if followeeId!= followerId:
            if followeeId in self.DB and followeeId in self.DB[followerId]["follow"].getList():
                return
            followee = Node(followeeId) # i should add it in when i come back to followerId and his posts to his feeds
            if followerId in self.DB:
                self.DB[followerId]["follow"].append(followee)
            else :
                self.DB[followerId] = {"feeds": Stack(), "follow": Stack(), "posts": Stack()}
                self.DB[followerId]["follow"].append(followee)
        # now i should add all posts :
            if followeeId in self.DB:
                lst = self.DB[followeeId]["posts"].getListWithT()
                self.DB[followerId]["feeds"].addingStack(lst)
            else :
                self.DB[followeeId] = {"feeds": Stack(), "follow": Stack(), "posts": Stack()}


    def unfollow(self, followerId: int, followeeId: int) -> None:

       if followerId in self.DB and followeeId!= followerId:
           if followeeId in self.DB:
                s = set(self.DB[followeeId]["posts"].getList())
                NewHead = Node(0)
                Nv = NewHead
                current =  self.DB[followerId]["feeds"].head
                while current :
                    if current.val in s:
                        current = current.next
                    else: break
                Nv.next = current

                if current :
                    Nv = Nv.next
                    current = current.next
                    while current :
                        next_node = current.next
                        if current.val not in s:
                            Nv.next = current
                            Nv = Nv.next
                        current = next_node
                    Nv.next = None
                    self.DB[followerId]["feeds"].head = NewHead.next
                    self.DB[followerId]["feeds"].last = Nv
                else :
                    self.DB[followerId]["feeds"].head = None
                    self.DB[followerId]["feeds"].last = None
                self.DB[followerId]["follow"].remove(followeeId)






### The test runner

Same replay style as Min Stack: a list of operation names and a list of
argument lists, replayed in order against your class. Run this cell; don't
edit it.

In [63]:
def run(ops, args):
    """Replay LeetCode-style (ops, args) against Twitter, collect outputs."""
    out, tw = [], None
    for op, a in zip(ops, args):
        if op == "Twitter":
            tw = Twitter()
            out.append(None)
        else:
            out.append(getattr(tw, op)(*a))
    return out


In [64]:
# tests
TESTS = [
    # the LeetCode example
    (["Twitter","postTweet","getNewsFeed","follow","postTweet","getNewsFeed","unfollow","getNewsFeed"],
     [[],[1,5],[1],[1,2],[2,6],[1],[1,2],[1]],
     [None,None,[5],None,None,[6,5],None,[5]]),

    # empty feed: never tweeted, follows nobody
    (["Twitter","getNewsFeed"],
     [[],[1]],
     [None,[]]),

    # 12 tweets from one user -> only the 10 most recent, newest first
    (["Twitter"] + ["postTweet"]*12 + ["getNewsFeed"],
     [[]] + [[1, i] for i in range(1, 13)] + [[1]],
     [None] + [None]*12 + [[12,11,10,9,8,7,6,5,4,3]]),

    # question 4's trap: follow yourself, unfollow yourself -> own tweets must survive
    (["Twitter","postTweet","follow","unfollow","getNewsFeed"],
     [[],[1,101],[1,1],[1,1],[1]],
     [None,None,None,None,[101]]),

    # question 5's trap: unfollow someone you never followed -> must not crash
    (["Twitter","postTweet","unfollow","getNewsFeed"],
     [[],[1,7],[1,2],[1]],
     [None,None,None,[7]]),

    # interleaved posts from three users, feed merges by global time
    (["Twitter","postTweet","postTweet","postTweet","postTweet","follow","follow","getNewsFeed","getNewsFeed"],
     [[],[1,10],[2,20],[1,11],[3,30],[1,2],[1,3],[1],[2]],
     [None,None,None,None,None,None,None,[30,11,20,10],[20]]),

    # following is one-way: 2 follows 1, but 1 does not see 2's tweets
    (["Twitter","follow","postTweet","postTweet","getNewsFeed","getNewsFeed"],
     [[],[2,1],[1,1],[2,2],[1],[2]],
     [None,None,None,None,[1],[2,1]]),

    # 15 tweets spread over three users, 1 follows both others -> cap at 10, newest first
    (["Twitter"] + ["postTweet"]*15 + ["follow","follow","getNewsFeed"],
     [[]] + [[(i % 3) + 1, 100 + i] for i in range(15)] + [[1,2],[1,3],[1]],
     [None] + [None]*15 + [None,None,[114,113,112,111,110,109,108,107,106,105]]),
]

for ops, args, expected in TESTS:
    got = run(ops, args)
    print(f"{'OK  ' if got == expected else 'FAIL'} {got}")
    if got != expected:
        print(f"     want {expected}")


OK   [None, None, [5], None, None, [6, 5], None, [5]]
OK   [None, []]
OK   [None, None, None, None, None, None, None, None, None, None, None, None, None, [12, 11, 10, 9, 8, 7, 6, 5, 4, 3]]
OK   [None, None, None, None, [101]]
OK   [None, None, None, [7]]
OK   [None, None, None, None, None, None, None, [30, 11, 20, 10], [20]]
OK   [None, None, None, None, [1], [2, 1]]
OK   [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, [114, 113, 112, 111, 110, 109, 108, 107, 106, 105]]


In [ ]:
# extra test - the gap LeetCode found that the suite above missed.
# Test 7 opens with follow(2,1) on a completely EMPTY object, so a guard that
# checks the wrong id still short-circuits and survives. Here user 1 exists and
# user 2 does not: the FOLLOWER's key is missing while the followee's key is
# present, so only a guard on the follower holds.
EXTRA = [
    (["Twitter","postTweet","follow","getNewsFeed","getNewsFeed"],
     [[],[1,5],[2,1],[2],[1]],
     [None,None,None,[5],[5]]),
]

for ops, args, expected in EXTRA:
    got = run(ops, args)
    print(f"{'OK  ' if got == expected else 'FAIL'} {got}")
    if got != expected:
        print(f"     want {expected}")


## V2 - the professional version (and why your memory score was 5.65%)

Your accepted solution is the **push model**: every post is copied into every
follower's stored feed, so reads are trivial (runtime beat 91%) and memory
pays for it (beat 5.65%). Count what one tweet costs you: a `Node` in the
author's `posts`, another in the author's `feeds`, plus one more in **each**
follower's `feeds` - and those feeds are never trimmed, so they keep every
tweet forever even though `getNewsFeed` only ever reads 10. A Python object
per copy is ~56 bytes plus pointers; the tuples below are one shared object
each.

The **pull model** below flips the trade: store each tweet exactly once, build
the feed when asked. Both classes pass all 9 tests.

- **`TwitterPull`** - route A. Storage: one dict of `(time, tweetId)` lists,
  one dict of sets. `postTweet`/`follow`/`unfollow` are O(1) one-liners; the
  work moves into `getNewsFeed`, which gathers at most 10 per source (the
  proof: a tweet in the global top 10 has <10 tweets newer than it anywhere,
  so it is inside its own author's last 10) and sorts. O(k log k) per feed
  with k sources - bounded, because the slice happens *before* the sort.
- **`TwitterHeap`** - route B, the one interviewers are fishing for. Same
  storage; the merge becomes #23 Merge k Sorted Lists. Seed a heap with the
  newest tweet of each source, pop the newest, push that source's next-older
  tweet, stop at 10. O(k + 10 log k) - it never touches the 11th tweet of any
  source. `heapq` is a *min*-heap, so `-time` makes newest come out first.

Three details worth stealing:

- `setdefault` on writes, `.get` on reads. Reading a feed must never create
  state - the same "`getNewsFeed` touches nothing" rule you asked about.
  (`defaultdict` would silently insert an empty entry on every read.)
- `discard` instead of `remove` - question 5, one word, no crash.
- Own tweets come from `{userId}` unioned in at read time, so no
  follow/unfollow call can reach them. Question 4's trap, closed by
  construction rather than by a guard.


In [ ]:
import heapq

class TwitterPull:
    """Route A - pull model: store once, build the feed on demand."""

    def __init__(self):
        self.time = 0
        self.tweets = {}    # userId -> [(time, tweetId)] in post order (oldest first)
        self.follows = {}   # userId -> set of followees

    def postTweet(self, userId: int, tweetId: int) -> None:
        self.tweets.setdefault(userId, []).append((self.time, tweetId))
        self.time += 1

    def follow(self, followerId: int, followeeId: int) -> None:
        if followerId != followeeId:
            self.follows.setdefault(followerId, set()).add(followeeId)

    def unfollow(self, followerId: int, followeeId: int) -> None:
        self.follows.get(followerId, set()).discard(followeeId)

    def getNewsFeed(self, userId: int) -> list:
        pool = []
        for uid in self.follows.get(userId, set()) | {userId}:
            pool.extend(self.tweets.get(uid, ())[-10:])
        pool.sort(reverse=True)
        return [tweetId for _, tweetId in pool[:10]]


class TwitterHeap:
    """Route B - same storage, but merge k sorted lists with a heap."""

    def __init__(self):
        self.time = 0
        self.tweets = {}
        self.follows = {}

    def postTweet(self, userId: int, tweetId: int) -> None:
        self.tweets.setdefault(userId, []).append((self.time, tweetId))
        self.time += 1

    def follow(self, followerId: int, followeeId: int) -> None:
        if followerId != followeeId:
            self.follows.setdefault(followerId, set()).add(followeeId)

    def unfollow(self, followerId: int, followeeId: int) -> None:
        self.follows.get(followerId, set()).discard(followeeId)

    def getNewsFeed(self, userId: int) -> list:
        heap = []
        for uid in self.follows.get(userId, set()) | {userId}:
            posts = self.tweets.get(uid)
            if posts:
                i = len(posts) - 1
                t, tweetId = posts[i]
                heap.append((-t, tweetId, uid, i))
        heapq.heapify(heap)

        feed = []
        while heap and len(feed) < 10:
            _, tweetId, uid, i = heapq.heappop(heap)
            feed.append(tweetId)
            if i:
                t, older = self.tweets[uid][i - 1]
                heapq.heappush(heap, (-t, older, uid, i - 1))
        return feed


In [ ]:
# V2 tests - run the same suites against both professional versions
for cls in (TwitterPull, TwitterHeap):
    Twitter = cls
    print(cls.__name__)
    for ops, args, expected in TESTS + EXTRA:
        out, tw = [], None
        for op, a in zip(ops, args):
            if op == "Twitter":
                tw = cls(); out.append(None)
            else:
                out.append(getattr(tw, op)(*a))
        print(f"  {'OK  ' if out == expected else 'FAIL'} {out[-1]}")
        if out != expected:
            print(f"       want {expected[-1]}")


## After it passes

- **Route B is a famous problem wearing a costume.** "Top 10 across k
  already-sorted lists" is #23 Merge k Sorted Lists, which is why LeetCode
  files this under *linked-list*. Build route B with `heapq` and re-run the
  same tests - the class is a drop-in replacement. What do you push onto the
  heap so the *newest* tweet comes out first, given `heapq` is a min-heap?
- **Name what made ordering free.** You never sorted inside one user's list -
  append order *was* time order. Where else this week did a structure hand
  you an ordering for free? (#105: preorder hands you the root for free;
  #155: the node under the top hands you the previous min for free.)
- **The clock is the design.** Everything hard here dissolved once one
  counter stamped every tweet. Compare: what dissolved #155? (One extra value
  carried per node.) Design problems are usually one well-placed piece of
  state, then bookkeeping.
- **Complexities, per operation:** write time and space for `postTweet`,
  `follow`, `unfollow`, `getNewsFeed` - route A and route B separately. Which
  three are O(1) in both routes?
- Siblings when you want more of this: #146 LRU Cache (the design problem),
  #23 Merge k Sorted Lists (route B, undisguised), #703 Kth Largest Element
  in a Stream (your first pure heap problem).


In [4]:
class Node:
    def __init__(self, val, next=None):
        self.val = val
        self.next = next

class Stack:
    def __init__(self):
        self.head = None
    def append(self,node:Node):
        if self.head is None:
            self.head = node
        else:
            node.next = self.head
            self.head = node
    def pop(self):
        if self.head :
            self.head = self.head.next
    def getList(self)->list:
        if self.head :
            curr = self.head
            lst = []
            while curr:
                lst.append(curr.val)
                curr = curr.next
            return lst
        return []

user = {
    1 : {"feeds" : Stack() , "follow" : Stack() , "posts"  : Stack() } ,
    2 : {"feeds" : Stack() , "follow" : Stack() , "posts"  : Stack() }
}
for u in user:
    print(user[u]["feeds"])